In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import (
    PROJECT_ROOT,
    DATA_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    WEATHER_DIR,
    SOIL_DIR,
    RAW_DIR,
    OUTPUT_DIR,
    PROCESSED_DATASET,
    WEATHER_FEATHER,
    MERGED_DATA_DIR,
    interim_csb_path,
)


# Corp plant cycle estimation

This notebook estimate the merged processed Crop Sequence Boundaries (CSB) acreage data with aggregated weather data to explore crop cycle and trend

In [ ]:
# Import libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Set plot style
sns.set_theme(style="whitegrid")
%matplotlib inline

# 1. Import data

In [ ]:
# Define Paths
# Note: Adjust relative paths if running from a different directory
csb_path = str(MERGED_DATA_DIR / "merged_CSB_total.csv")
weather_path = str(WEATHER_FEATHER)

# 1. Load CSB Data
if os.path.exists(csb_path):
    csb_df = pd.read_csv(csb_path)
    print(f"CSB Data Loaded: {csb_df.shape}")
else:
    print(f"Error: CSB file not found at {csb_path}")


# 2. Process & Aggregate Weather Data
# Goal: Create Annual-County level weather metrics (e.g., Total Precip during Growing Season)

In [ ]:

if 'weather_df' in locals():
    # Convert dates
    weather_df['start_date'] = pd.to_datetime(weather_df['start_date'])
    weather_df['Year'] = weather_df['start_date'].dt.year
    weather_df['Month'] = weather_df['start_date'].dt.month
    
    # Filter for Growing Season (April - October)
    # Adjust months as needed for specific crops
    growing_season = weather_df[weather_df['Month'].between(4, 10)]
    
    # Aggregate
    weather_annual = growing_season.groupby(['county_name', 'Year']).agg({
        'totr': 'sum',       # Total Rainfall
        'avgt': 'mean',      # Average Temperature
        'gdd_b10': 'sum',    # Growing Degree Days (Base 10C approx Base 50F)
        'mint': 'min',       # Minimum Temperature recorded
        'maxt': 'max'        # Maximum Temperature recorded
    }).reset_index()
    
    # Rename columns for merging
    weather_annual.rename(columns={
        'county_name': 'CNTY',
        'totr': 'Total_Precip_AprOct',
        'avgt': 'Avg_Temp_AprOct',
        'gdd_b10': 'GDD_AprOct',
        'mint': 'Min_Temp_AprOct',
        'maxt': 'Max_Temp_AprOct'
    }, inplace=True)
    
    print("Weather Aggregation Complete.")
    display(weather_annual.head())

In [ ]:
# 3.1 Feature Engineering: Seasonal Weather Metrics & Anomalies

if 'weather_df' in locals():
    # Define seasons
    planting_season = weather_df[weather_df['Month'].between(4, 5)]
    growing_season = weather_df[weather_df['Month'].between(6, 8)]
    harvest_season = weather_df[weather_df['Month'].between(9, 10)]
    
    # Helper function to aggregate
    def aggregate_season(df, prefix):
        agg = df.groupby(['county_name', 'Year']).agg({
            'totr': 'sum',
            'avgt': 'mean',
            'gdd_b10': 'sum',
            'maxt': lambda x: (x > 32.2).sum() # Count days > 90F (32.2C)
        }).reset_index()
        agg.rename(columns={
            'totr': f'{prefix}_Precip',
            'avgt': f'{prefix}_AvgTemp',
            'gdd_b10': f'{prefix}_GDD',
            'maxt': f'{prefix}_HeatDays'
        }, inplace=True)
        return agg

    # Aggregate by season
    plant_agg = aggregate_season(planting_season, 'Planting')
    grow_agg = aggregate_season(growing_season, 'Growing')
    harv_agg = aggregate_season(harvest_season, 'Harvest')
    
    # Merge all seasonal data
    seasonal_weather = pd.merge(plant_agg, grow_agg, on=['county_name', 'Year'], how='outer')
    seasonal_weather = pd.merge(seasonal_weather, harv_agg, on=['county_name', 'Year'], how='outer')
    
    # Rename for consistency
    seasonal_weather.rename(columns={'county_name': 'CNTY'}, inplace=True)
    
    # Calculate Anomalies (Deviation from 5-year rolling average)
    # We need to sort by Year to calculate rolling stats
    seasonal_weather.sort_values(['CNTY', 'Year'], inplace=True)
    
    metrics = ['Planting_Precip', 'Growing_GDD', 'Growing_Precip']
    for metric in metrics:
        # Calculate 5-year rolling mean (closed window, min_periods=1)
        seasonal_weather[f'{metric}_5y_Avg'] = seasonal_weather.groupby('CNTY')[metric].transform(
            lambda x: x.rolling(window=5, min_periods=1).mean().shift(1) # Shift to use PAST 5 years
        )
        # Calculate Deviation
        seasonal_weather[f'{metric}_Anomaly'] = seasonal_weather[metric] - seasonal_weather[f'{metric}_5y_Avg']
    
    print("Seasonal Weather & Anomalies Created.")
    display(seasonal_weather.head())

In [ ]:
# 4. Merge Datasets (Enhanced)

if 'csb_df' in locals() and 'seasonal_weather' in locals():
    # Merge CSB with Seasonal Weather
    merged_df = pd.merge(csb_df, seasonal_weather, on=['CNTY', 'Year'], how='inner')
    
    # 4.1 Lagged Variables (Crop Rotation)
    # We want to know the acreage of the SAME crop in the PREVIOUS year
    merged_df.sort_values(['CNTY', 'Crop_Name', 'Year'], inplace=True)
    merged_df['Acres_Lag1'] = merged_df.groupby(['CNTY', 'Crop_Name'])['CSBACRES'].shift(1)
    
    # 4.2 Soil Data (Placeholder - assuming static soil file exists or creating dummy)
    # In a real scenario, load soil.csv with columns ['CNTY', 'Percent_Prime_Farmland']
    # Here we will simulate it or skip if file doesn't exist
    soil_path = str(SOIL_DIR / "soil_features_NY.csv") # Example path
    if os.path.exists(soil_path):
        soil_df = pd.read_csv(soil_path)
        merged_df = pd.merge(merged_df, soil_df, on='CNTY', how='left')
    else:
        print("Soil data not found, skipping soil merge.")
    
    print(f"Merged Dataset Shape: {merged_df.shape}")
    display(merged_df.head())

# 3. Plant data analysis

In [ ]:
# ---------------------------------------------------------
# 3. Plant Cycle Analysis & Distribution
# ---------------------------------------------------------
# Discussion on Studying Plant Cycle with Annual Data:
# 1. Long-term Trends: Since data is annual, we look for trends over years rather than seasonal cycles within a year.
# 2. Inter-annual Variability: We can observe year-to-year fluctuations which may be driven by climate or market cycles.
# 3. Distribution Analysis: Agricultural data is often highly skewed (many small values, few large ones). 
#    Log-transformation is essential to study the underlying distribution and variance.

# 3.1 Time Series Plot of CSBACRES
# We aggregate by Year to see the total trend
plt.figure(figsize=(12, 6))
annual_acres = csb_df.groupby('Year')['CSBACRES'].sum().reset_index()
sns.lineplot(data=annual_acres, x='Year', y='CSBACRES', marker='o', linewidth=2.5)
plt.title('Total CSB Acreage Over Time (Annual Cycle)', fontsize=14)
plt.ylabel('Total Acres')
plt.grid(True, linestyle='--', alpha=0.7)
# plt.savefig('csb_acreage_time_series.png')
# print("Saved time series plot to csb_acreage_time_series.png")
plt.show()

# 3.2 Distribution Analysis: Histogram of CSBACRES vs Log(CSBACRES)
plt.figure(figsize=(14, 6))

# Raw CSBACRES
plt.subplot(1, 2, 1)
sns.histplot(csb_df['CSBACRES'], bins=50, kde=True, color='skyblue')
plt.title('Distribution of CSBACRES (Raw)', fontsize=12)
plt.xlabel('Acres')
plt.ylabel('Frequency')

# Log-Transformed CSBACRES
# We use log1p (log(x+1)) to handle zero values safely
plt.subplot(1, 2, 2)
log_acres = np.log1p(csb_df['CSBACRES'])
sns.histplot(log_acres, bins=50, kde=True, color='green')
plt.title('Distribution of Log(CSBACRES)', fontsize=12)
plt.xlabel('Log(Acres + 1)')
plt.ylabel('Frequency')

plt.tight_layout()
# plt.savefig('csb_acreage_distribution.png')
# print("Saved distribution plot to csb_acreage_distribution.png")
plt.show()

# Summary Statistics
print("\nSummary Statistics (Raw):")
print(csb_df['CSBACRES'].describe())
print("\nSummary Statistics (Log-Transformed):")
print(log_acres.describe())


In [ ]:
# 5. Exploratory Analysis: Correlation

if 'merged_df' in locals():
    # Select numeric columns for correlation
    # We focus on major crops to avoid noise from minor ones
    major_crops = ['Corn', 'Soybeans', 'Alfalfa', 'Winter Wheat']
    subset = merged_df[merged_df['Crop_Name'].isin(major_crops)]
    
    # Pivot to get crops as columns if we want to see crop-specific correlations vs weather
    # Or just correlate CSBACRES with Weather for each crop group
    
    for crop in major_crops:
        crop_data = subset[subset['Crop_Name'] == crop]
        if crop_data.empty: continue
            
        print(f"\n--- Correlation for {crop} ---")
        cols = ['CSBACRES', 'Acres_Lag1', 'Planting_Precip', 'Planting_Precip_Anomaly', 'Growing_GDD', 'Growing_GDD_Anomaly']
        corr = crop_data[cols].corr()
        display(corr)
        
        # Simple Scatter Plot
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        sns.scatterplot(data=crop_data, x='Planting_Precip', y='CSBACRES')
        plt.title(f'{crop} Acres vs Planting Precip')
        
        plt.subplot(1, 2, 2)
        sns.scatterplot(data=crop_data, x='Growing_GDD', y='CSBACRES')
        plt.title(f'{crop} Acres vs Growing GDD')
        plt.tight_layout()
        plt.show()

In [ ]:
# # Export Combined Weather Data
# output_path = str(MERGED_DATA_DIR / "CSB_weather_combined.feather")

# if 'weather_annual' in locals():
#     weather_annual.to_feather(output_path)
#     print(f"Data exported to {output_path}")
# elif 'seasonal_weather' in locals():
#     seasonal_weather.to_feather(output_path)
#     print(f"Data exported to {output_path}")
# else:
#     print("No weather dataframe found to export.")